# SemanticDraw SD1.5 + LCM Full1073 Metric Run trên Kaggle

Notebook này chạy end-to-end baseline **SemanticDraw** với full manifest COCO val2017 hợp lệ theo protocol SD1.5 512x512:

`Ours/data_manifests/coco_val2017_multidiffusion_coco_all_512x512_all.jsonl`

Full manifest hiện tại có **1073 sample**. Mỗi sample tương ứng một ảnh COCO hợp lệ sau bước filter của dataloader/manifest.

**Model được chạy:** Stable Diffusion 1.5, checkpoint `runwayml/stable-diffusion-v1-5`.

**Sampler / tăng tốc được chạy:** LCM. Trong source baseline, `SemanticDrawPipeline(sd_version="1.5")` tự thay scheduler thành `LCMScheduler` và tự gắn LoRA `latent-consistency/lcm-lora-sdv1-5`.

Vì vậy trong notebook ta sẽ không thấy dòng gọi `LCMScheduler(...)` trực tiếp. Phần đó nằm trong:

`Baseline/semantic-draw-main/src/model/pipeline_semantic_draw.py`

Notebook chỉ gọi `SemanticDrawPipeline`; pipeline baseline tự load SD1.5 + LCM-LoRA + LCMScheduler.

Sau khi sinh ảnh xong, notebook chạy phần đo metric:

`FID`, `IS`, `CLIP(fg)`, `CLIP(bg)`, `Time(s)`

Kết quả metric được lưu tại:

`/kaggle/working/semanticdraw_full1073_metrics/`

## Ghi chú quan trọng

Đây là bản chạy **full 1073 sample**, không phải smoke/mini test. Với `BATCH_SIZE = 8`, dataloader có khoảng 135 batch; tuy nhiên generation của baseline vẫn chạy tuần tự từng ảnh trong từng batch.

`MAX_DISPLAY_RESULTS = 8` chỉ giới hạn số preview hiển thị trong notebook để Kaggle không bị nặng. Ảnh vẫn được sinh và lưu đủ theo 1073 record trong manifest.

Output generation được lưu tại:

`/kaggle/working/semanticdraw_full1073_outputs/`

Nếu muốn chạy nhanh để debug, dùng notebook Mini128 cũ hoặc đổi `RUN_MANIFEST` về mini32/smoke trong cell cấu hình. Không nên đổi trực tiếp notebook full nếu mục tiêu là benchmark chính thức.

Mục tiêu notebook này là chạy toàn bộ đường benchmark SD1.5 + LCM trên manifest chính: COCO -> dataloader -> masks/prompts -> baseline SemanticDraw -> ảnh sinh ra -> metrics.


## 0. Yêu cầu Kaggle

- Bật GPU trong `Settings -> Accelerator -> GPU`.
- Bật Internet để tải COCO, Stable Diffusion 1.5, LCM LoRA, Inception weights và CLIP weights.
- Nếu Hugging Face yêu cầu quyền truy cập, thêm Kaggle Secret tên `HF_TOKEN` hoặc set biến môi trường `HF_TOKEN`.
- Full 1073 sẽ lâu hơn mini128 rõ rệt. Nên chạy thử Mini128 trước để chắc chắn pipeline, COCO download và metric đều hoạt động.
- Khi chạy full, giữ `MAX_DISPLAY_RESULTS` nhỏ để notebook không phình RAM vì hiển thị quá nhiều ảnh.


In [ ]:
# Cài các thư viện cần thiết cho generation + metric evaluation.
# Không cài lại torch để tránh làm lệch môi trường GPU mặc định của Kaggle.
# Quan trọng: gỡ torchao. Một số Kaggle image có torchao==0.10.0;
# peft mới thấy torchao nhưng yêu cầu >0.16.0, gây lỗi khi load LCM LoRA.
import sys
import subprocess

packages = [
    "diffusers>=0.30.0",
    "transformers>=4.44.0",
    "accelerate",
    "peft",
    "huggingface_hub",
    "safetensors",
    "sentencepiece",
    "protobuf",
    "einops",
    "pycocotools",
    "matplotlib",
    "tqdm",
    "pandas>=2.0",
    "open-clip-torch>=2.24.0",
    "torch-fidelity>=0.3.0",
    "torchmetrics>=1.4",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
print("[OK] Dependencies are ready. torchao is removed to avoid PEFT LoRA compatibility errors.")


In [ ]:
# Clone repo nếu notebook chưa nằm trong repo clone.
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/GOx9-P/AnchorDraw.git"
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

def is_repo_root(path: Path) -> bool:
    return (
        (path / "Baseline" / "semantic-draw-main" / "src").exists()
        and (path / "Ours" / "test_sets" / "manifests" / "smoke").exists()
    )

def find_repo_root() -> Path | None:
    starts = [
        Path.cwd(),
        Path.cwd() / "AnchorDraw",
        WORK_DIR / "AnchorDraw",
        WORK_DIR / "AnchorDraw" / "AnchorDraw",
        WORK_DIR / "anchor_draw",
    ]
    checked = set()
    for start in starts:
        for path in [start, *start.parents]:
            path = path.resolve()
            if path in checked:
                continue
            checked.add(path)
            if is_repo_root(path):
                return path
    return None

REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    clone_target = WORK_DIR / "AnchorDraw"
    if not clone_target.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_target)], check=True)
    REPO_ROOT = find_repo_root()

assert REPO_ROOT is not None and is_repo_root(REPO_ROOT), "Không tìm thấy repo root sau khi clone."
print(f"[OK] Repo root: {REPO_ROOT}")

In [ ]:
# Cấu hình full manifest 1073 cho benchmark SD1.5 + LCM.
from pathlib import Path

RUN_MANIFEST = REPO_ROOT / "Ours" / "data_manifests" / "coco_val2017_multidiffusion_coco_all_512x512_all.jsonl"
COCO_ROOT = Path(os.environ.get("COCO_ROOT", "/kaggle/working/COCO"))
OUTPUT_DIR = Path("/kaggle/working/semanticdraw_full1073_outputs")
MASK_CACHE_DIR = Path("/kaggle/working/semanticdraw_mask_cache")
METRICS_OUTPUT_DIR = Path("/kaggle/working/semanticdraw_full1073_metrics")

# Model theo baseline SD1.5. Có thể đổi sang một SD1.5-compatible checkpoint nếu cần.
MODEL_ID = "runwayml/stable-diffusion-v1-5"

# BATCH_SIZE chỉ là số sample mỗi lượt dataloader load.
# Tổng số ảnh sinh ra = số record trong manifest vì cell generation sẽ lặp qua toàn bộ loader.
TARGET_SIZE = (512, 512)
BATCH_SIZE = 8
BASE_SEED = 2024
BOOTSTRAP_STEPS = 1
MASK_STD = 0.0
MASK_STRENGTH = 1.0
PREPROCESS_MASK_COVER_ALPHA = 0.0
MASK_TYPE = "discrete"
NEGATIVE_PROMPT = ""

# Chỉ hiển thị 8 preview để Kaggle notebook không bị nặng.
# Ảnh vẫn được sinh và lưu đủ theo full manifest 1073.
MAX_DISPLAY_RESULTS = 8

METRIC_NAMES = ("fid", "is", "clip_fg", "clip_bg", "time")
METRIC_BATCH_SIZE = 8
CLIP_BATCH_SIZE = 16
IS_SPLITS = 10
METRICS_REPORT_PREFIX = "semanticdraw_sd15_lcm_full1073_metrics"

assert RUN_MANIFEST.exists(), f"Missing manifest: {RUN_MANIFEST}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MASK_CACHE_DIR.mkdir(parents=True, exist_ok=True)
METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Manifest: {RUN_MANIFEST}")
print(f"[OK] COCO root: {COCO_ROOT}")
print(f"[OK] Output dir: {OUTPUT_DIR}")
print(f"[OK] Metrics output dir: {METRICS_OUTPUT_DIR}")
print(f"[OK] Batch size: {BATCH_SIZE}")


In [ ]:
# Tải COCO val2017 nếu Kaggle runtime chưa có sẵn dữ liệu.
# Lưu ý: trên một số Kaggle runtime, HTTPS của images.cocodataset.org có thể lỗi SSL.
# Vì vậy cell này ưu tiên HTTP official COCO và có nhiều fallback download.
import ssl
import urllib.request
import zipfile

COCO_ROOT.mkdir(parents=True, exist_ok=True)

VAL_ZIP_URLS = [
    "http://images.cocodataset.org/zips/val2017.zip",
    "https://images.cocodataset.org/zips/val2017.zip",
]
ANN_ZIP_URLS = [
    "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
    "https://images.cocodataset.org/annotations/annotations_trainval2017.zip",
]
val_zip = COCO_ROOT / "val2017.zip"
ann_zip = COCO_ROOT / "annotations_trainval2017.zip"


def run_download_command(cmd: list[str]) -> bool:
    try:
        subprocess.run(cmd, check=True)
        return True
    except Exception as exc:
        print(f"[WARN] Download command failed: {' '.join(cmd[:2])} -> {exc}")
        return False


def download_file(urls: list[str], dst: Path) -> None:
    if dst.exists() and dst.stat().st_size > 0:
        print(f"[SKIP] Already downloaded: {dst.name}")
        return

    last_error = None
    for url in urls:
        print(f"[DOWNLOAD] {url}")

        # 1) wget fallback. --no-check-certificate handles Kaggle SSL hostname mismatch.
        if run_download_command(["wget", "-c", "--no-check-certificate", "-O", str(dst), url]):
            if dst.exists() and dst.stat().st_size > 0:
                return

        # 2) curl fallback. -k disables certificate verification; -L follows redirects.
        if run_download_command(["curl", "-L", "-k", "--retry", "3", "-o", str(dst), url]):
            if dst.exists() and dst.stat().st_size > 0:
                return

        # 3) urllib fallback with unverified SSL context only for this public dataset download.
        try:
            context = ssl._create_unverified_context()
            with urllib.request.urlopen(url, context=context, timeout=120) as response:
                with dst.open("wb") as f:
                    f.write(response.read())
            if dst.exists() and dst.stat().st_size > 0:
                return
        except Exception as exc:
            last_error = exc
            print(f"[WARN] urllib failed for {url}: {exc}")

    raise RuntimeError(
        f"Cannot download {dst.name}. Last error: {last_error}. "
        "Check Kaggle Internet setting, or attach COCO val2017 as a Kaggle Dataset and set COCO_ROOT."
    )


def unzip_if_missing(zip_path: Path, marker_path: Path) -> None:
    if marker_path.exists():
        print(f"[SKIP] Already extracted: {marker_path}")
        return
    print(f"[UNZIP] {zip_path.name}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(COCO_ROOT)


download_file(VAL_ZIP_URLS, val_zip)
download_file(ANN_ZIP_URLS, ann_zip)
unzip_if_missing(val_zip, COCO_ROOT / "val2017" / "000000000139.jpg")
unzip_if_missing(ann_zip, COCO_ROOT / "annotations" / "instances_val2017.json")

assert (COCO_ROOT / "val2017").exists(), "Missing COCO val2017 images."
assert (COCO_ROOT / "annotations" / "instances_val2017.json").exists(), "Missing instances_val2017.json."
assert (COCO_ROOT / "annotations" / "captions_val2017.json").exists(), "Missing captions_val2017.json."
print("[OK] COCO val2017 is ready.")


In [ ]:
# Import dataloader của Ours và baseline SemanticDrawPipeline.
import sys
import importlib.util
import json
import time

import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

OURS_SRC = REPO_ROOT / "Ours" / "src"
BASELINE_SRC = REPO_ROOT / "Baseline" / "semantic-draw-main" / "src"

sys.path.insert(0, str(OURS_SRC))
from data import COCORegionConfig, build_coco_region_dataloader, batch_item_to_semanticdraw_inputs
from data.visualize import make_mask_overlay

# Load trực tiếp file pipeline_semantic_draw.py để tránh import toàn bộ model/__init__.py
# vì các file SDXL/SD3 có thể cần dependency khác không dùng trong smoke SD1.5.
sys.path.insert(0, str(BASELINE_SRC))
pipeline_path = BASELINE_SRC / "model" / "pipeline_semantic_draw.py"
spec = importlib.util.spec_from_file_location("pipeline_semantic_draw", pipeline_path)
pipeline_module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(pipeline_module)
SemanticDrawPipeline = pipeline_module.SemanticDrawPipeline

print("[OK] Imports are ready.")

In [ ]:
# Tạo dataloader cho full manifest 1073.
config = COCORegionConfig(
    coco_root=COCO_ROOT,
    split="val2017",
    instances_json=COCO_ROOT / "annotations" / "instances_val2017.json",
    captions_json=COCO_ROOT / "annotations" / "captions_val2017.json",
    manifest_path=RUN_MANIFEST,
    profile="multidiffusion_coco_all",
    model_family="sd15",
    target_size=TARGET_SIZE,
    return_image=True,
    cache_resized_masks=True,
    cache_dir=MASK_CACHE_DIR,
    batch_size=BATCH_SIZE,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False,
)

loader = build_coco_region_dataloader(config, shuffle=False, drop_last=False)
dataset_size = len(loader.dataset)
num_batches = len(loader)
preview_batch = next(iter(loader))

print(f"[OK] Manifest records: {dataset_size}")
print(f"[OK] Dataloader batches: {num_batches} batch(es) x up to {BATCH_SIZE} sample(s)")
print(f"[OK] First batch size: {len(preview_batch['sample_ids'])}")
print(f"[OK] First batch masks shape: {tuple(preview_batch['masks'].shape)}  # (B, Pmax, C, H, W)")
print("First batch sample IDs:")
for sample_id in preview_batch["sample_ids"]:
    print(" -", sample_id)


## 1. Chuẩn hóa input cho SemanticDraw

Manifest lưu foreground object masks/prompts và background caption riêng. Demo baseline của tác giả thường đưa background vào như region mask đầu tiên.

Ở đây ta tạo:

`background_mask = 1 - union(foreground_masks)`

Sau đó input cho SemanticDraw là:

`prompts = [COCO caption] + foreground_prompts`

`masks = [background_mask] + foreground_masks`


In [ ]:
def md_escape(text: object) -> str:
    return str(text).replace("\n", " ").replace("|", "\\|")

def seed_everything(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_semanticdraw_payload(batch: dict, index: int) -> dict:
    item = batch_item_to_semanticdraw_inputs(batch, index)
    fg_masks = item["masks"].float().cpu()  # (p, 1, H, W)
    fg_union = fg_masks.sum(dim=0, keepdim=True).clamp(0, 1)
    background_mask = (1.0 - fg_union).clamp(0, 1)
    all_masks = torch.cat([background_mask, fg_masks], dim=0)

    prompts = [item["background_prompt"], *item["prompts"]]
    negative_prompts = [NEGATIVE_PROMPT for _ in prompts]
    metadata = item["metadata"]

    return {
        "sample_id": item["metadata"]["sample_id"],
        "image_id": item["metadata"]["image_id"],
        "file_name": item["metadata"]["file_name"],
        "height": item["height"],
        "width": item["width"],
        "prompts": prompts,
        "negative_prompts": negative_prompts,
        "foreground_prompts": item["prompts"],
        "category_names": metadata["category_names"],
        "annotation_ids": metadata["annotation_ids"],
        "area_ratios": metadata["area_ratios"],
        "foreground_masks": fg_masks,
        "all_masks": all_masks,
        "metadata": metadata,
    }

def display_smoke_result(payload: dict, original: Image.Image, overlay: Image.Image, generated: Image.Image, elapsed: float, generated_path: Path) -> None:
    rows = ["| Region | Prompt | Annotation | Area ratio |", "|---|---|---:|---:|"]
    rows.append(f"| Background | {md_escape(payload['prompts'][0])} | - | - |")
    for label, prompt, ann_id, area in zip(payload["category_names"], payload["foreground_prompts"], payload["annotation_ids"], payload["area_ratios"]):
        rows.append(f"| {md_escape(label)} | {md_escape(prompt)} | {ann_id} | {float(area):.4f} |")

    display(Markdown(
        f"### `{payload['sample_id']}`\n"
        f"- image_id: `{payload['image_id']}`\n"
        f"- file: `{payload['file_name']}`\n"
        f"- generated path: `{generated_path}`\n"
        f"- elapsed: `{elapsed:.2f}s`\n\n"
        + "\n".join(rows)
    ))

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(original)
    axes[0].set_title("COCO original resized")
    axes[1].imshow(overlay)
    axes[1].set_title("Foreground mask overlay")
    axes[2].imshow(generated)
    axes[2].set_title("SemanticDraw generated")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("[OK] Helper functions are ready.")

In [ ]:
# Login Hugging Face nếu có token trong Kaggle Secret hoặc biến môi trường.
def maybe_login_to_huggingface() -> None:
    token = os.environ.get("HF_TOKEN")
    if token is None:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if token:
        from huggingface_hub import login
        login(token=token)
        print("[OK] Hugging Face token loaded.")
    else:
        print("[INFO] No HF_TOKEN found. Public/gated model access depends on your Hugging Face permissions.")

assert torch.cuda.is_available(), "Kaggle runtime chưa bật GPU. Hãy bật Accelerator = GPU rồi chạy lại."
device = torch.device("cuda:0")
dtype = torch.float16

maybe_login_to_huggingface()
print(f"[OK] GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Guard chống lỗi torchao/PEFT trước khi load LCM LoRA.
# Nếu bạn vừa gặp lỗi torchao ở cell load pipeline, hãy restart Kaggle session rồi Run All.
import sys
import importlib
import importlib.util

importlib.invalidate_caches()

if "torchao" in sys.modules:
    raise RuntimeError(
        "torchao is already imported in this Python session. Restart the Kaggle session, run the dependency cell, then Run All. "
        "SemanticDraw SD1.5 + LCM LoRA does not need torchao for this smoke test."
    )

if importlib.util.find_spec("torchao") is not None:
    raise RuntimeError(
        "torchao is still installed/importable in this runtime. Run the dependency cell, then restart the Kaggle session and Run All. "
        "SemanticDraw SD1.5 + LCM LoRA does not need torchao for this smoke test."
    )

print("[OK] torchao is not importable; PEFT should skip torchao LoRA dispatch.")


In [ ]:
# Load baseline SemanticDraw SD1.5 pipeline.
# has_i2t=False để không tải BLIP-2 vì COCO caption đã là background prompt.
seed_everything(BASE_SEED)
smd = SemanticDrawPipeline(
    device=device,
    dtype=dtype,
    sd_version="1.5",
    hf_key=MODEL_ID,
    has_i2t=False,
    default_mask_std=MASK_STD,
    default_mask_strength=MASK_STRENGTH,
    default_preprocess_mask_cover_alpha=PREPROCESS_MASK_COVER_ALPHA,
    mask_type=MASK_TYPE,
)

if hasattr(smd.pipe, "enable_attention_slicing"):
    smd.pipe.enable_attention_slicing()

print("[OK] SemanticDrawPipeline is ready.")

In [ ]:
# Kiểm tra trực tiếp notebook đang dùng sampler/scheduler nào.
print("Scheduler:", type(smd.scheduler).__name__)
print("Pipeline scheduler:", type(smd.pipe.scheduler).__name__)
print("Default inference steps:", smd.default_num_inference_steps)
print("Default guidance scale:", smd.default_guidance_scale)

loaded_adapters = getattr(smd.pipe, "get_active_adapters", None)
if callable(loaded_adapters):
    print("Active LoRA adapters:", loaded_adapters())
else:
    print("Active LoRA adapters: check not available in this diffusers version")

assert type(smd.scheduler).__name__ == "LCMScheduler", "Expected LCM sampler via LCMScheduler for SD1.5 smoke test."


In [ ]:
# Chạy generation cho mọi sample trong manifest và hiển thị kết quả.
summary = []
global_index = 0

for batch_index, batch in enumerate(loader):
    print(f"[BATCH] {batch_index + 1}/{len(loader)} - {len(batch['sample_ids'])} sample(s)")

    for local_index, sample_id in enumerate(batch["sample_ids"]):
        payload = make_semanticdraw_payload(batch, local_index)
        original = batch["images"][local_index].resize((payload["width"], payload["height"]), Image.Resampling.BILINEAR)
        overlay = make_mask_overlay(original, payload["foreground_masks"], payload["category_names"], alpha=0.45)

        seed = BASE_SEED + global_index
        seed_everything(seed)

        tic = time.perf_counter()
        generated = smd(
            prompts=payload["prompts"],
            negative_prompts=payload["negative_prompts"],
            masks=payload["all_masks"].to(device=device, dtype=torch.float32),
            mask_stds=MASK_STD,
            mask_strengths=MASK_STRENGTH,
            height=payload["height"],
            width=payload["width"],
            bootstrap_steps=BOOTSTRAP_STEPS,
            preprocess_mask_cover_alpha=PREPROCESS_MASK_COVER_ALPHA,
            do_blend=False,
        )
        elapsed = time.perf_counter() - tic

        stem = f"{global_index:04d}_{payload['sample_id']}"
        generated_path = OUTPUT_DIR / f"{stem}_generated.png"
        overlay_path = OUTPUT_DIR / f"{stem}_overlay.png"
        generated.save(generated_path)
        overlay.save(overlay_path)

        summary.append({
            "index": global_index,
            "batch_index": batch_index,
            "local_index": local_index,
            "sample_id": payload["sample_id"],
            "image_id": payload["image_id"],
            "file_name": payload["file_name"],
            "seed": seed,
            "num_regions_including_background": len(payload["prompts"]),
            "elapsed_sec": elapsed,
            "generated_path": str(generated_path),
            "overlay_path": str(overlay_path),
        })

        should_display = MAX_DISPLAY_RESULTS is None or global_index < MAX_DISPLAY_RESULTS
        if should_display:
            display_smoke_result(payload, original, overlay, generated, elapsed, generated_path)

        global_index += 1
        torch.cuda.empty_cache()

summary_path = OUTPUT_DIR / "generation_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

display(Markdown(
    f"## Done\n"
    f"Generated `{len(summary)}` image(s) from `{dataset_size}` manifest record(s). "
    f"Summary saved to `{summary_path}`."
))
summary[:5]


## 2. Đo metric sau generation

Phần dưới dùng ảnh đã sinh trong `OUTPUT_DIR` và `generation_summary.json` để tính:

`FID`, `IS`, `CLIP(fg)`, `CLIP(bg)`, `Time(s)`

Trước khi đo metric, notebook giải phóng pipeline diffusion khỏi GPU để có chỗ load Inception/CLIP. Nếu muốn generate lại ảnh sau bước này, hãy chạy lại cell load `SemanticDrawPipeline` trước.


In [ ]:
# Giải phóng VRAM trước khi load Inception/CLIP cho metric.
import gc

for var_name in ("smd", "generated", "payload", "overlay", "original", "batch", "loader", "preview_batch"):
    if var_name in globals():
        del globals()[var_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

print("[OK] Released generation objects before metric evaluation.")


In [ ]:
# Đo FID, IS, CLIP(fg), CLIP(bg), Time(s) trên full output vừa generate.
from metrics import MetricEvaluationConfig, run_evaluation, write_metrics_report
import pandas as pd
import math

generation_summary_path = OUTPUT_DIR / "generation_summary.json"
assert generation_summary_path.exists(), f"Missing generation summary: {generation_summary_path}"

metric_device = "cuda:0" if torch.cuda.is_available() else "cpu"
metric_config = MetricEvaluationConfig(
    manifest_path=RUN_MANIFEST,
    coco_root=COCO_ROOT,
    generated_dir=OUTPUT_DIR,
    generation_summary=generation_summary_path,
    output_dir=METRICS_OUTPUT_DIR,
    model_family="sd15",
    target_size=TARGET_SIZE,
    metrics=METRIC_NAMES,
    batch_size=METRIC_BATCH_SIZE,
    num_workers=0,
    pin_memory=False,
    device=metric_device,
    clip_batch_size=CLIP_BATCH_SIZE,
    is_splits=IS_SPLITS,
)

metric_report = run_evaluation(metric_config)
metrics_json, metrics_csv = write_metrics_report(
    metric_report,
    METRICS_OUTPUT_DIR,
    prefix=METRICS_REPORT_PREFIX,
)

values = metric_report["metrics"]

def fmt(value: object, digits: int = 4) -> str:
    if value is None:
        return "-"
    try:
        value = float(value)
        if math.isnan(value):
            return "-"
        return f"{value:.{digits}f}"
    except Exception:
        return str(value)

metrics_table = pd.DataFrame([
    {"Metric": "FID↓", "Value": fmt(values.get("fid"))},
    {"Metric": "IS↑", "Value": fmt(values.get("is_mean"))},
    {"Metric": "IS std", "Value": fmt(values.get("is_std"))},
    {"Metric": "CLIP(fg)↑", "Value": fmt(values.get("clip_fg_x100"))},
    {"Metric": "CLIP(bg)↑", "Value": fmt(values.get("clip_bg_x100"))},
    {"Metric": "Time(s)↓", "Value": fmt(values.get("time_mean_sec"))},
    {"Metric": "Total time(s)", "Value": fmt(values.get("time_total_sec"))},
])

display(Markdown(
    f"## Metric Done\n"
    f"- evaluated: `{metric_report['num_evaluated']}` / `{metric_report['num_manifest_records']}` samples\n"
    f"- missing generated images: `{metric_report['num_missing_generated']}`\n"
    f"- metrics JSON: `{metrics_json}`\n"
    f"- metrics CSV: `{metrics_csv}`"
))
display(metrics_table)

metric_report
